# Continual Image Classification — final GPU benchmark
Run from top to bottom after selecting a GPU runtime. No API key is required; never paste credentials into this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import zipfile

BUNDLE = Path('/content/drive/MyDrive/continual_dl_colab_bundle.zip')
PROJECT = Path('/content/continual-image-classification')
assert BUNDLE.exists(), f'Upload bundle first: {BUNDLE}'
if not PROJECT.exists():
    with zipfile.ZipFile(BUNDLE) as archive:
        archive.extractall('/content')
print(PROJECT)

In [ ]:
%cd /content/continual-image-classification
!pip install -q -e .

In [ ]:
import os
import torch
from pathlib import Path

assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))
!python scripts/verify_dataset.py data/manifests/train.csv data/manifests/val.csv data/manifests/test.csv

drive_outputs = Path('/content/drive/MyDrive/continual_dl_outputs')
drive_outputs.mkdir(parents=True, exist_ok=True)
local_outputs = Path('/content/continual-image-classification/outputs')
if local_outputs.is_symlink():
    assert local_outputs.resolve() == drive_outputs.resolve()
elif local_outputs.exists():
    raise RuntimeError(f'Remove or rename existing local output path: {local_outputs}')
else:
    os.symlink(drive_outputs, local_outputs, target_is_directory=True)
print('Persistent outputs:', drive_outputs)

In [ ]:
# Preview the 18-run matrix. Existing summaries in Drive will be skipped.
!python scripts/run_benchmark.py --common-config configs/colab.yaml --device cuda --dry-run --strategies ncm naive replay lwf ewc joint

In [ ]:
# Long-running cell. It is safe to rerun after a disconnect.
!python scripts/run_benchmark.py --common-config configs/colab.yaml --device cuda --strategies ncm naive replay lwf ewc joint

In [ ]:
import pandas as pd
!python scripts/summarize_results.py
display(pd.read_csv('outputs/comparison.csv'))
display(pd.read_csv('outputs/per_seed_results.csv'))